# Run Each Generator Separately

This notebook calls `run.py` once per pipeline using `--only`. Run only the cell for the model you want to test, because some cells can use paid API calls.

## Important Notes and Reproducibility Scope

This notebook serves as an executable technical appendix for the project entrypoint `master_connections/run.py`. Each section runs one pipeline with `--only`, so models can be tested independently without running the full weighted ensemble.

### What this notebook covers
- Per-pipeline generation using `run.py --only <pipeline>`
- Dry-run registration checks for all pipelines
- Optional Abuzar AI agentic-mode run (`--agent`)
- Environment setup for `.venv`, `.env` loading, and subprocess `python` compatibility
- Offline HuggingFace mode for Kevin Fresh when model cache is already present

### Requirements and inputs
- Project repository containing `master_connections/run.py`
- API keys in `master_connections/.env` as needed by selected pipelines
  - Burak: Anthropic/OpenAI (for some labeling/validation)
  - Adreama: OpenAI
  - Kevin Fresh: OpenAI + local SentenceTransformer cache
  - Abuzar AI: Anthropic
  - Abuzar NLP: local datasets (no API key required for normal runs)
- For Kevin Fresh offline mode, `all-mpnet-base-v2` should be downloaded at least once

### Reproducibility notes
- LLM calls and random sampling are stochastic; exact puzzle boards may differ across reruns
- Reproducibility here is procedure-level (same commands/settings/environment), not exact board identity
- Rerun the setup cell whenever the kernel restarts

### Outputs and interpretation
- Accepted puzzles are written to `master_connections/output/generated_puzzles.json`
- `Return code: 0` means the command completed, not necessarily that a puzzle was accepted
- Always inspect logs for outcomes such as `OK`, `duplicate`, `None`, or `Gave up after attempts`

### Optional web export (outside notebook)
```bash
cd master_connections
python scripts/export_web_puzzles.py
```


In [13]:
# Environment summary (appendix metadata)
from pathlib import Path
import os
import platform
import subprocess
import sys

print('Platform:         ', platform.platform())
print('Python version:   ', sys.version.splitlines()[0])
print('Python executable:', sys.executable)
print('CWD:              ', Path.cwd())

for key in [
    'ANTHROPIC_API_KEY',
    'OPENAI_API_KEY',
    'HF_HUB_OFFLINE',
    'TRANSFORMERS_OFFLINE',
    'MASTER_CONNECTIONS_DIR',
    'NYT_CONNECTIONS_ROOT',
]:
    val = os.environ.get(key)
    shown = 'set' if (val and 'KEY' in key) else (val if val is not None else 'unset')
    print(f'{key}:', shown)

# Lightweight package versions (best effort)
def pkg_ver(pkg):
    try:
        out = subprocess.check_output([sys.executable, '-m', 'pip', 'show', pkg], text=True)
        for line in out.splitlines():
            if line.startswith('Version:'):
                return line.split(':', 1)[1].strip()
    except Exception:
        return 'unknown'
    return 'unknown'

for pkg in [
    'anthropic', 'openai', 'gensim', 'nltk', 'scikit-learn',
    'sentence-transformers', 'transformers', 'torch',
]:
    print(f'{pkg}:', pkg_ver(pkg))


Platform:          macOS-15.7.3-arm64-arm-64bit
Python version:    3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
Python executable: /opt/anaconda3/bin/python
CWD:               /Users/burakdonbekci/Library/Containers/net.whatsapp.WhatsApp/Data/tmp/documents/B4D81AB4-1347-447C-AAA4-EB5EADB9A593
ANTHROPIC_API_KEY: set
OPENAI_API_KEY: set
HF_HUB_OFFLINE: 1
TRANSFORMERS_OFFLINE: 1
MASTER_CONNECTIONS_DIR: unset
NYT_CONNECTIONS_ROOT: unset
anthropic: 0.96.0
openai: 2.32.0
gensim: 4.3.3
nltk: 3.9.1
scikit-learn: 1.5.1
sentence-transformers: 5.4.1
transformers: 5.5.4
torch: 2.11.0


In [14]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import tempfile

# Auto-detect master_connections by walking up from CWD, then trying env/manual fallbacks.
start = Path.cwd().resolve()
PROJECT_DIR = None

# 1) Walk up from current directory.
for candidate in [start, *start.parents]:
    if (candidate / "run.py").exists():
        PROJECT_DIR = candidate
        break
    nested = candidate / "master_connections"
    if (nested / "run.py").exists():
        PROJECT_DIR = nested
        break

# 2) Optional explicit overrides (set either in the notebook or shell env).
# Example:
# MANUAL_PROJECT_DIR = Path.home() / "Desktop" / "NYT-Connections-Game" / "master_connections"
MANUAL_PROJECT_DIR = None

overrides = [
    MANUAL_PROJECT_DIR,
    os.environ.get("MASTER_CONNECTIONS_DIR"),
    os.environ.get("NYT_CONNECTIONS_ROOT"),
    Path.home() / "Desktop" / "NYT-Connections-Game" / "master_connections",
    Path.home() / "Documents" / "NYT-Connections-Game" / "master_connections",
]

if PROJECT_DIR is None:
    for raw in overrides:
        if not raw:
            continue
        p = Path(raw).expanduser().resolve()
        # If root repo path was provided, append master_connections.
        if p.name != "master_connections" and (p / "master_connections").exists():
            p = p / "master_connections"
        if (p / "run.py").exists():
            PROJECT_DIR = p
            break

assert PROJECT_DIR is not None and (PROJECT_DIR / "run.py").exists(), (
    "Could not find master_connections/run.py. "
    f"Start dir: {start}. "
    "Set MANUAL_PROJECT_DIR in this cell or export MASTER_CONNECTIONS_DIR."
)

def load_env_file(path):
    """Small .env loader so this notebook does not require python-dotenv."""
    path = Path(path)
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ[key] = value
    return True

env_loaded = load_env_file(PROJECT_DIR / ".env")

VENV_PYTHON = PROJECT_DIR / ".venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON if VENV_PYTHON.exists() else sys.executable)

# Some adapters call subprocesses with the command name "python".
# VS Code/Jupyter kernels may not put that exact executable on PATH,
# so create a temporary shim named "python" that points to this interpreter.
python_bin = str(Path(PYTHON).parent)
shim_dir = Path(tempfile.gettempdir()) / "nyt_connections_python_shim"
shim_dir.mkdir(parents=True, exist_ok=True)
shim_python = shim_dir / "python"
if shim_python.exists() or shim_python.is_symlink():
    shim_python.unlink()
# Use a shell wrapper, not a symlink. A symlink outside .venv can make Python
# lose its virtualenv context and fail to find installed packages.
shim_python.write_text(f"#!/bin/sh\nexec {shlex.quote(PYTHON)} \"$@\"\n", encoding="utf-8")
shim_python.chmod(0o755)
os.environ["PATH"] = str(shim_dir) + os.pathsep + python_bin + os.pathsep + os.environ.get("PATH", "")

# Kevin Fresh uses a HuggingFace SentenceTransformer model. After the model is
# downloaded once, keep HF/Transformers offline so it does not fail on optional
# network checks during notebook runs.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"Project dir: {PROJECT_DIR}")
print(f"Python:      {PYTHON}")
print(f"PATH first:  {shim_dir}")
print(f"Shim python: {shim_python} runs {PYTHON}")
print(f"HF offline:  {os.environ.get('HF_HUB_OFFLINE')}")
print(f".env loaded:  {env_loaded}")
print(f"OpenAI key:  {'set' if os.environ.get('OPENAI_API_KEY') else 'missing'}")

Project dir: /Users/burakdonbekci/Desktop/NYT-Connections-Game/master_connections
Python:      /opt/anaconda3/bin/python
PATH first:  /var/folders/nk/fgwtf8nj44lc1h5xzg1pdn480000gn/T/nyt_connections_python_shim
Shim python: /var/folders/nk/fgwtf8nj44lc1h5xzg1pdn480000gn/T/nyt_connections_python_shim/python runs /opt/anaconda3/bin/python
HF offline:  1
.env loaded:  True
OpenAI key:  set


In [15]:
def run_model(pipeline, n=1, *, agent=False, dry_run=False, preview=False, check=False):
    """Run one pipeline through master_connections/run.py and stream output."""
    cmd = [PYTHON, "run.py", "--only", pipeline, "--n", str(n)]
    if agent:
        cmd.append("--agent")
    if dry_run:
        cmd.append("--dry-run")
    if preview:
        cmd.append("--preview")

    print("$", shlex.join(cmd))
    print()

    run_env = os.environ.copy()
    run_env["HF_HUB_OFFLINE"] = "1"
    run_env["TRANSFORMERS_OFFLINE"] = "1"
    if "shim_dir" in globals():
        run_env["PATH"] = str(shim_dir) + os.pathsep + run_env.get("PATH", "")

    print(f"HF_HUB_OFFLINE={run_env.get('HF_HUB_OFFLINE')}")
    print(f"TRANSFORMERS_OFFLINE={run_env.get('TRANSFORMERS_OFFLINE')}")
    print()

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=run_env,
    )

    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)

    returncode = process.wait()
    print(f"\nReturn code: {returncode}")

    if check and returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd, output="".join(output_lines))

    return {"pipeline": pipeline, "returncode": returncode, "output": "".join(output_lines)}

## Optional: Registration Check

This checks whether each pipeline can register. It does not generate puzzles or make API calls.

In [16]:
PIPELINES = ["burak", "adreama", "kevin_fresh", "abuzar_nlp", "abuzar_ai"]

registration_results = {}
for pipeline in PIPELINES:
    print("=" * 80)
    registration_results[pipeline] = run_model(pipeline, dry_run=True)

$ /opt/anaconda3/bin/python run.py --only burak --n 1 --dry-run

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Common words : 9,474
Full dict    : 234,375  Short: 4,618  CMU: 123,455
Utilities loaded.
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/a

## Burak

Runs only Burak's pipeline.

In [8]:
burak_result = run_model("burak", n=1)

$ /opt/anaconda3/bin/python run.py --only burak --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Common words : 9,474
Full dict    : 234,375  Short: 4,618  CMU: 123,455
Utilities loaded.
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.8.0 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/l

## Adreama

Runs only Adreama's pipeline. Requires `OPENAI_API_KEY`.

In [9]:
adreama_result = run_model("adreama", n=1)

$ /opt/anaconda3/bin/python run.py --only adreama --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: adreama
DedupStore: 288 puzzles loaded from /Users/burakdonbekci/Desktop/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['adreama']
  Weights   : {'adreama': 1.0}
  Store     : 288 puzzles so far
  NYT archive guard: ON (path=/Users/burakdonbekci/Desktop/NYT-Connections-Game/abuzar_AI-Connections/data/nyt_blocklist.json ready=True boards=1029 group_sets=1033)
  Attempt 1: [adreama]... Config loaded.
Memory loaded.
  Words used   : 0
  Categories   : 0
  Boards saved : 0
  Puzzles in CSV: 0
Total webs: 48
48 category webs loaded.
Generator loaded.
None (16.2s)
  Attempt 2: [adreama]... OK (3.8s) → puzzle #289
  Source  : adreama
  YELLOW : types of materials                       ['FABRIC', 'WOOD', 'METAL', 'PLASTIC']
  GREEN  : things associated with sound             ['ECHO', 'CHIME', 'RING', 'WHISPER']
  BLUE   : ___

## Kevin Fresh

Runs only Kevin Fresh. Requires `OPENAI_API_KEY`.

In [10]:
kevin_fresh_result = run_model("kevin_fresh", n=1)

$ /opt/anaconda3/bin/python run.py --only kevin_fresh --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: kevin_fresh
DedupStore: 289 puzzles loaded from /Users/burakdonbekci/Desktop/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['kevin_fresh']
  Weights   : {'kevin_fresh': 1.0}
  Store     : 289 puzzles so far
  NYT archive guard: ON (path=/Users/burakdonbekci/Desktop/NYT-Connections-Game/abuzar_AI-Connections/data/nyt_blocklist.json ready=True boards=1029 group_sets=1033)
  Attempt 1: [kevin_fresh]... duplicate — exact board duplicate
  Attempt 2: [kevin_fresh]... duplicate — duplicate yellow group (words): ['SOCIABILITY', 'SOCIABLY', 'SOCIALITY', 'SOCIALLY']
  Gave up after 2 attempts.

Pipeline stats:
  abuzar_nlp          : 32
  kevin_fresh         : 11
  adreama             : 103
  abuzar_ai           : 8
  burak               : 135

Return code: 0


## Abuzar NLP

Runs only Abuzar NLP. This uses local datasets and does not require an API key.

In [11]:
abuzar_nlp_result = run_model("abuzar_nlp", n=1)

$ /opt/anaconda3/bin/python run.py --only abuzar_nlp --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_nlp
DedupStore: 289 puzzles loaded from /Users/burakdonbekci/Desktop/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_nlp']
  Weights   : {'abuzar_nlp': 1.0}
  Store     : 289 puzzles so far
  NYT archive guard: ON (path=/Users/burakdonbekci/Desktop/NYT-Connections-Game/abuzar_AI-Connections/data/nyt_blocklist.json ready=True boards=1029 group_sets=1033)
  Attempt 1: [abuzar_nlp]... OK (0.1s) → puzzle #290
  Source  : abuzar_nlp
  YELLOW : Dwellings                                ['CASTLE', 'FARMHOUSE', 'RESIDENCE', 'VILLA']
  GREEN  : Wheeled Vehicles                         ['CAR', 'ELECTRIC', 'SALOON', 'TRUCK']
  BLUE   : Criminal Terms                           ['PEDDLER', 'PUNK', 'PUSHER', 'THIEF']
  PURPLE : SUN ___                                  ['BURN', 'RISE', 'SET', 'SHINE']


Pipeline stat

## Abuzar AI

Runs only Abuzar AI's standard fresh-puzzle path. Requires `ANTHROPIC_API_KEY`.

In [12]:
abuzar_ai_result = run_model("abuzar_ai", n=1)

$ /opt/anaconda3/bin/python run.py --only abuzar_ai --n 1

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_ai
DedupStore: 290 puzzles loaded from /Users/burakdonbekci/Desktop/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_ai']
  Weights   : {'abuzar_ai': 1.0}
  Store     : 290 puzzles so far
  NYT archive guard: ON (path=/Users/burakdonbekci/Desktop/NYT-Connections-Game/abuzar_AI-Connections/data/nyt_blocklist.json ready=True boards=1029 group_sets=1033)
  Attempt 1: [abuzar_ai]... OK (112.3s) → puzzle #291
  Source  : abuzar_ai
  YELLOW : Common Words That Are Also Greek Letters ['DELTA', 'SIGMA', 'OMEGA', 'ALPHA']
  GREEN  : Words After DEAD                         ['LOCK', 'LINE', 'WOOD', 'PAN']
  BLUE   : Common Words That Are Also Heraldic Term ['AZURE', 'GULES', 'SABLE', 'OR']
  PURPLE : Also a Type of Logical Fallacy           ['STRAW', 'SLIPPERY', 'APPEAL', 'CIRCULAR']


Pipeline stats:
  abuzar_nl

## Optional: Abuzar AI Agentic Mode

Runs Abuzar AI through `run.py --agent`. This can be slower and can use more API calls. Note: This model is expensive ($0.2 - $1 per call) to run, we suggest running it only a few times to test.

In [34]:
abuzar_ai_agent_result = run_model("abuzar_ai", n=1, agent=True)

$ '/Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/.venv/bin/python' run.py --only abuzar_ai --n 1 --agent

HF_HUB_OFFLINE=1
TRANSFORMERS_OFFLINE=1

Registered: abuzar_ai (agentic mode)
DedupStore: 245 puzzles loaded from /Users/abuzarkhudaverdiyeva/Documents/New project/external_repos/NYT-Connections-Game/master_connections/output/generated_puzzles.json
MasterGenerator ready.
  Pipelines : ['abuzar_ai']
  Weights   : {'abuzar_ai': 1.0}
  Store     : 245 puzzles so far
  Attempt 1: [abuzar_ai]...   [abuzar_ai] subprocess failed (1): Starting bank size: 68
Target new puzzles: 1
Safety limits: max_attempts=1, max_empty_batches=1, max_review_rounds=1, strategy=standard

Batch 1: requesting 1 accepted puzzle(s)
Accepted this batch: 0
Rejected this batch: 2
Bank size: 68

Stopped after 1 consecutive empty batch(es) to avoid extra API spend.

Stopped after 1 batches with 0/1 new puzzles.
None (44.8s)
  Attempt 2: [abuzar_ai]...   [abuz